## Repomix & UTF-8 Code Aggregator

Скрипт для автоматической сборки исходного кода проекта (MetaTrader MQL4/MQL5, Python, C++ и Jupyter Notebooks) в единый текстовый файл с гарантированной очисткой и пересохранением в кодировку **UTF-8**.

Предназначен для удобного экспорта всей кодовой базы проекта и ее последующей передачи в языковые модели (LLM / ChatGPT / Claude).

---

### 🚀 Возможности

* **Автоматическое имя файла:** формируется на основе названия целевой папки (`<Имя_Папки>-code-summary-utf8.txt`).
* **Гибкая фильтрация:** включение только нужных расширений файлов и исключение служебных каталогов.
* **Защита от закольцовывания:** итоговые файлы автоматически добавляются в список исключений `--ignore`.
* **Исправление кодировок:** решает проблему битых символов при сборке файлов с кодировками ANSI/Windows-1251 (актуально для MQL4/MQL5).

---

### 🛠 Требования и установка

1. **Python** 3.8 или выше.
2. **Node.js** (поставляется вместе с утилитой `npx`):
   ```bash
   node -v
   npx -v

In [13]:
import fnmatch
import subprocess
from pathlib import Path

# ================= НАСТРОЙКИ ПАРАМЕТРОВ =================
path_dir = r"C:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform"
include = ["trading_conditions/Creation/ipynb_files/hybrid_spread_markup.ipynb",
    "Calculation_of_Paid_Spread_per_Trade_samples.ipynb", "hybrid_spread_markup.ipynb", "Analysis_Sessions_Quotes.ipynb",
    "ipynb_files/creation_symbolsGroups.ipynb"]
ignore = ["files/**", "**/uploading_symbols.ipynb"]

"""
include = ["**/*.mq5", "**/*.py", "**/*.cpp", "**/*.mqh", "**/*.ipynb"]
ignore = [
    "files/**", "*.csv", "migration_errors/**", "own_platform/**",
    "GitHub_through_Python/**", "input_data/**", "output_data/**",
    "ipynb_files-code-summary.txt", "ipynb_files-code-summary-utf8.txt",
    "**/Analysis_MT5_configuration_file.ipynb", "**/uploading_symbols.ipynb", "trading_conditions/Creation/ipynb_files/hybrid_spread_markup.ipynb",
    "Calculation_of_Paid_Spread_per_Trade_samples.ipynb", "hybrid_spread_markup.ipynb", "Analysis_Sessions_Quotes.ipynb",
    "ipynb_files/creation_symbolsGroups.ipynb"
]"""
# =======================================================

# 1. Проверка и настройка путей
TARGET_DIR = Path(path_dir).resolve()
if TARGET_DIR.is_file():
    TARGET_DIR = TARGET_DIR.parent

if not TARGET_DIR.exists():
    raise FileNotFoundError(f"Указанный каталог не найден: {TARGET_DIR}")

dir_name = TARGET_DIR.name
OUTPUT_FILENAME = f"{dir_name}-code-summary.txt"
OUTPUT_UTF8_FILENAME = f"{dir_name}-code-summary-utf8.txt"

output_path = TARGET_DIR / OUTPUT_FILENAME
output_utf8_path = TARGET_DIR / OUTPUT_UTF8_FILENAME

all_ignore = ignore + [OUTPUT_FILENAME, OUTPUT_UTF8_FILENAME]

def format_size(size_bytes: int) -> str:
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 * 1024:
        return f"{size_bytes / 1024:.2f} KB"
    else:
        return f"{size_bytes / (1024 * 1024):.2f} MB"

def matches_patterns(rel_str: str, patterns: list) -> bool:
    for pat in patterns:
        if fnmatch.fnmatch(rel_str, pat) or fnmatch.fnmatch(rel_str, f"**/{pat}"):
            return True
        prefix = pat.split('*')[0].rstrip('/')
        if prefix and (rel_str == prefix or rel_str.startswith(prefix + '/')):
            return True
    return False

# 2. Сборка проекта через Repomix
repomix_command = [
    "npx", "repomix",
    "--style", "plain",
    "--include", ",".join(include),
    "--ignore", ",".join(all_ignore),
    "--output", str(output_path)
]

print(f"Запуск Repomix для директории: {TARGET_DIR}...")
subprocess.run(repomix_command, cwd=TARGET_DIR, check=True, shell=True)

# 3. Очистка, подсчет слов и пересохранение в UTF-8
print("Очистка и пересохранение файла в UTF-8...")
with open(output_path, 'r', encoding='utf-8', errors='ignore') as f:
    content = f.read()

# Подсчет количества слов и строк
word_count = len(content.split())
line_count = len(content.splitlines())

with open(output_utf8_path, 'w', encoding='utf-8') as f:
    f.write(content)

# 4. Анализ вошедших файлов и вывод статистики
collected_files = []
for file_path in TARGET_DIR.rglob('*'):
    if not file_path.is_file():
        continue
    
    rel_str = file_path.relative_to(TARGET_DIR).as_posix()
    
    if matches_patterns(rel_str, include) and not matches_patterns(rel_str, all_ignore):
        collected_files.append((rel_str, file_path.stat().st_size))

collected_files.sort(key=lambda x: x[1], reverse=True)

total_source_size = sum(size for _, size in collected_files)
final_file_size = output_utf8_path.stat().st_size

# ====================================================
# Ширина столбца для пути (увеличена с 42 до 80)
PATH_WIDTH = 80
TOTAL_WIDTH = PATH_WIDTH + 26  # Авторасчет общей ширины таблицы

print("\n" + "=" * TOTAL_WIDTH)
print("📊 ИТОГОВАЯ СТАТИСТИКА СБОРКИ")
print("=" * TOTAL_WIDTH)
print(f"Путь к итоговому файлу: {output_utf8_path}")
print(f"Размер итогового UTF-8 файла: {format_size(final_file_size)} ({final_file_size:,} байт)")
print(f"Количество слов в файле:      {word_count:,}")
print(f"Количество строк в файле:     {line_count:,}")
print(f"Всего файлов обработано:      {len(collected_files)}")
print("-" * TOTAL_WIDTH)
print(f"{'№':<4} | {'Относительный путь к файлу':<{PATH_WIDTH}} | {'Размер':<10} | {'Доля'}")
print("-" * TOTAL_WIDTH)

for idx, (rel_path, f_size) in enumerate(collected_files, 1):
    pct = (f_size / total_source_size * 100) if total_source_size > 0 else 0
    # Обрезка с усечением под ширину PATH_WIDTH
    display_path = rel_path if len(rel_path) <= PATH_WIDTH else "..." + rel_path[-(PATH_WIDTH - 3):]
    print(f"{idx:<4} | {display_path:<{PATH_WIDTH}} | {format_size(f_size):<10} | {pct:>5.1f}%")

print("=" * TOTAL_WIDTH)

Запуск Repomix для директории: C:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform...
Очистка и пересохранение файла в UTF-8...

📊 ИТОГОВАЯ СТАТИСТИКА СБОРКИ
Путь к итоговому файлу: C:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\own_platform-code-summary-utf8.txt
Размер итогового UTF-8 файла: 1.59 MB (1,667,942 байт)
Количество слов в файле:      108,962
Количество строк в файле:     36,186
Всего файлов обработано:      5
----------------------------------------------------------------------------------------------------------
№    | Относительный путь к файлу                                                       | Размер     | Доля
----------------------------------------------------------------------------------------------------------
1    | trading_conditions/Creation/ipynb_files/hybrid_spread_markup.ipynb               | 1.16 MB    |  28.7%
2    | ipynb_files/hybrid_spread_markup.ipynb                                           | 1.1

In [ ]:
import fnmatch
import subprocess
from pathlib import Path

# ================= НАСТРОЙКИ ПАРАМЕТРОВ =================
path_dir = r"C:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform"
include = ["**/*.mq5", "**/*.py", "**/*.cpp", "**/*.mqh", "**/*.ipynb"]
ignore = [
    "files/**", "migration_errors/**", "own_platform/**",
    "GitHub_through_Python/**", "input_data/**", "output_data/**",
    "ipynb_files-code-summary.txt", "ipynb_files-code-summary-utf8.txt",
    "**/Analysis_MT5_configuration_file.ipynb", "**/uploading_symbols.ipynb"
]
# =======================================================

# 1. Проверка и настройка путей
TARGET_DIR = Path(path_dir).resolve()
if TARGET_DIR.is_file():
    TARGET_DIR = TARGET_DIR.parent

if not TARGET_DIR.exists():
    raise FileNotFoundError(f"Указанный каталог не найден: {TARGET_DIR}")

dir_name = TARGET_DIR.name
OUTPUT_FILENAME = f"{dir_name}-code-summary.txt"
OUTPUT_UTF8_FILENAME = f"{dir_name}-code-summary-utf8.txt"

output_path = TARGET_DIR / OUTPUT_FILENAME
output_utf8_path = TARGET_DIR / OUTPUT_UTF8_FILENAME

all_ignore = ignore + [OUTPUT_FILENAME, OUTPUT_UTF8_FILENAME]

# Helper-функции для форматирования и проверки масок
def format_size(size_bytes: int) -> str:
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 * 1024:
        return f"{size_bytes / 1024:.2f} KB"
    else:
        return f"{size_bytes / (1024 * 1024):.2f} MB"

def matches_patterns(rel_str: str, patterns: list) -> bool:
    for pat in patterns:
        if fnmatch.fnmatch(rel_str, pat) or fnmatch.fnmatch(rel_str, f"**/{pat}"):
            return True
        prefix = pat.split('*')[0].rstrip('/')
        if prefix and (rel_str == prefix or rel_str.startswith(prefix + '/')):
            return True
    return False

# 2. Сборка проекта через Repomix
repomix_command = [
    "npx", "repomix",
    "--style", "plain",
    "--include", ",".join(include),
    "--ignore", ",".join(all_ignore),
    "--output", str(output_path)
]

print(f"Запуск Repomix для директории: {TARGET_DIR}...")
subprocess.run(repomix_command, cwd=TARGET_DIR, check=True, shell=True)

# 3. Очистка и пересохранение в UTF-8
print("Очистка и пересохранение файла в UTF-8...")
with open(output_path, 'r', encoding='utf-8', errors='ignore') as f:
    content = f.read()

with open(output_utf8_path, 'w', encoding='utf-8') as f:
    f.write(content)

# 4. Анализ вошедших файлов и вывод статистики
collected_files = []
for file_path in TARGET_DIR.rglob('*'):
    if not file_path.is_file():
        continue
    
    rel_str = file_path.relative_to(TARGET_DIR).as_posix()
    
    if matches_patterns(rel_str, include) and not matches_patterns(rel_str, all_ignore):
        collected_files.append((rel_str, file_path.stat().st_size))

# Сортировка файлов по убыванию размера
collected_files.sort(key=lambda x: x[1], reverse=True)

total_source_size = sum(size for _, size in collected_files)
final_file_size = output_utf8_path.stat().st_size

print("\n" + "=" * 70)
print("📊 ИТОГОВАЯ СТАТИСТИКА СБОРКИ")
print("=" * 70)
print(f"Путь к итоговому файлу: {output_utf8_path}")
print(f"Размер итогового UTF-8 файла: {format_size(final_file_size)} ({final_file_size:,} байт)")
print(f"Всего файлов обработано: {len(collected_files)}")
print("-" * 70)
print(f"{'№':<4} | {'Относительный путь к файлу':<42} | {'Размер':<10} | {'Доля'}")
print("-" * 70)

for idx, (rel_path, f_size) in enumerate(collected_files, 1):
    pct = (f_size / total_source_size * 100) if total_source_size > 0 else 0
    # Обрезка слишком длинных путей для красивого вывода в консоль
    display_path = rel_path if len(rel_path) <= 42 else "..." + rel_path[-39:]
    print(f"{idx:<4} | {display_path:<42} | {format_size(f_size):<10} | {pct:>5.1f}%")

print("=" * 70)

In [ ]:
import os
import subprocess
from pathlib import Path

path_dir = r"C:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform"
include = ["**/*.mq5", "**/*.py", "**/*.cpp", "**/*.mqh", "**/*.ipynb"]
ignore = ["files/**", "*.csv", "migration_errors/**", "own_platform/**", "GitHub_through_Python/**", "input_data/**", "output_data/**", "ipynb_files-code-summary.txt", "ipynb_files-code-summary-utf8.txt"]

# ================= НАСТРОЙКА ПУТЕЙ =================
# Укажите только абсолютный путь к целевой папке с проектом:
TARGET_DIR = Path(path_dir).resolve()

# Автоматическое формирование имен файлов на основе имени каталога
dir_name = TARGET_DIR.name  # Имя крайнего каталога (например, 'MyForexProject')

OUTPUT_FILENAME = f"{dir_name}-code-summary.txt"
OUTPUT_UTF8_FILENAME = f"{dir_name}-code-summary-utf8.txt"

output_path = TARGET_DIR / OUTPUT_FILENAME
output_utf8_path = TARGET_DIR / OUTPUT_UTF8_FILENAME
# ====================================================

# 1. Сборка проекта через repomix
repomix_command = [
    "npx", "repomix",
    "--style", "plain",
    "--include", ",".join(include),
    "--ignore", ",".join(ignore + [OUTPUT_FILENAME, OUTPUT_UTF8_FILENAME]),
    "--output", str(output_path)
]

print(f"Запуск Repomix для директории: {TARGET_DIR}...")
subprocess.run(repomix_command, cwd=TARGET_DIR, check=True, shell=True)

# 2. Очистка и пересохранение в UTF-8
print("Очистка и пересохранение файла в UTF-8...")
with open(output_path, 'r', encoding='utf-8', errors='ignore') as f:
    content = f.read()

with open(output_utf8_path, 'w', encoding='utf-8') as f:
    f.write(content)

print(f"Готово! Итоговый файл сохранен по пути:\n{output_utf8_path}")

3. Убедитесь, что `repomix` доступен через `npx` (установка не требуется, `npx` запустит его автоматически).

---

## ⚙️ Настройка скрипта

Все настройки находятся в блоке **`НАСТРОЙКИ ПАРАМЕТРОВ`** в начале файла:

```python
# Абсолютный путь к каталогу вашего проекта
path_dir = r"C:\Projects\MyForexProject"

# Маски файлов, которые нужно включить в сборку
include = ["**/*.mq5", "**/*.py", "**/*.cpp", "**/*.mqh", "**/*.ipynb"]

# Папки и файлы, которые нужно пропустить
ignore = ["files/**", "migration_errors/**", "own_platform/**", "GitHub_through_Python/**"]

```

---

## 📖 Как это работает

1. **Инициализация путей:** Скрипт получает имя конечного каталога (например, `MyForexProject`) и формирует целевые пути.
2. **Запуск Repomix:** Выполняет команду `npx repomix` в указанной директории с флагом `--style plain`.
3. **Конвертация в UTF-8:** Читает сгенерированный файл с флагом `errors='ignore'` для удаления поврежденных байтов и записывает итоговый результат в чистом формате UTF-8.

---

## 📄 Результат работы

В папке проекта создаются два файла:

1. `<Имя_Папки>-code-summary.txt` — исходный запуск Repomix.
2. `<Имя_Папки>-code-summary-utf8.txt` — очищенный итоговый файл в UTF-8 (готовый для загрузки в LLM).

```

```